In [1]:
!pip -qqq install pip --progress-bar off
!pip -qqq install 'crewai[tools]'==0.28.8 --progress-bar off
!pip -qqq install langchain-groq==0.1.3 --progress-bar off
!pip -qqq install duckduckgo-search==5.3.0 --progress-bar off

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi-cli 0.0.4 requires typer>=0.12.3, but you have typer 0.9.4 which is incompatible.


In [3]:
import os
import re
from datetime import datetime

import pandas as pd
import requests
from crewai import Agent, Crew, Process, Task
from crewai_tools import tool
from langchain.agents import load_tools
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq


def format_response(response: str) -> str:
    entries = re.split(r"(?<=]), (?=\[)", response)
    return [entry.strip("[]") for entry in entries]


os.environ["GROQ_API_KEY"] = ''

## Tools

In [4]:
search_tool = DuckDuckGoSearchResults(backend="news", num_results=10)

In [5]:
response = search_tool.run("Bitcoin")
format_response(response)

["snippet: Bitcoin's outlook remains positive as experts discuss political catalysts, demand growth, and enhanced investment transparency., title: How Trump's re-election and market dynamics could boost bitcoin, link: https://www.msn.com/en-us/money/markets/how-trumps-re-election-and-market-dynamics-could-boost-bitcoin/ar-BB1qDKqZ, date: 2024-07-25T22:12:08+00:00, source: TheStreet on MSN.com",
 'snippet: There is a larger police presence around the Music City Center where more than 20,000 people are attending the Bitcoin Conference., title: Extra security measures at Bitcoin Conference, link: https://www.wsmv.com/video/2024/07/26/extra-security-measures-bitcoin-conference/, date: 2024-07-26T07:37:00+00:00, source: WSMV',
 'snippet: Republican nominee and former U.S. President Donald Trump once blasted cryptocurrency, calling it a "scam." Now, he is headlining one of the industry\'s biggest conferences. Trump, who made the comment in 2021,, title: Trump, Republicans court crypto votes 

In [6]:
human_tools = load_tools(["human"])

In [10]:
def get_daily_closing_prices(ticker) -> pd.DataFrame:
    url = f'https://www.alphavantage.co/query?function=DIGITAL_CURRENCY_DAILY&symbol={ticker}&market=USD&apikey=HCNFZ47YS50B2N2W'
    response = requests.get(url)
    data = response.json()
    price_data = data["Time Series (Digital Currency Daily)"]
    daily_close_prices = {
        date: prices["4. close"] for (date, prices) in price_data.items()
    }

    df = pd.DataFrame.from_dict(daily_close_prices, orient="index", columns=["price"])
    df.index = pd.to_datetime(df.index)
    df["price"] = pd.to_numeric(df["price"])

    return df

In [11]:
price_df = get_daily_closing_prices("BTC")
price_df.head()

,price
2024-07-26,65925.83
2024-07-25,65795.42
2024-07-24,65366.40
2024-07-23,65939.24
2024-07-22,67558.22


In [12]:
text_output = []
for date, row in price_df.head(10).iterrows():
    text_output.append(f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}")
formatted_text = "\n".join(text_output)
print(formatted_text)

2024-07-26 - 65925.83
2024-07-25 - 65795.42
2024-07-24 - 65366.40
2024-07-23 - 65939.24
2024-07-22 - 67558.22
2024-07-21 - 68181.28
2024-07-20 - 67163.83
2024-07-19 - 66707.61
2024-07-18 - 63982.86
2024-07-17 - 64085.81


In [13]:
@tool("price tool")
def cryptocurrency_price_tool(ticker_symbol: str) -> str:
    """Get daily closing price for a given cryptocurrency ticker symbol for the previous 60 days"""
    price_df = get_daily_closing_prices(ticker_symbol)
    text_output = []
    for date, row in price_df.head(60).iterrows():
        text_output.append(f"{date.strftime('%Y-%m-%d')} - {row['price']:.2f}")
    return "\n".join(text_output)

In [14]:
@tool("search tool")
def cryptocurrency_news_tool(ticker_symbol: str) -> str:
    """Get news for a given cryptocurrency ticker symbol"""
    return search_tool.run(ticker_symbol + " cryptocurrency")

## Llama 3 with Groq

In [15]:
llm = ChatGroq(temperature=0, model_name="llama3-70b-8192")

In [16]:
%%time
system = "You are experienced Machine Learning & AI Engineer."
human = "{text}"
prompt = ChatPromptTemplate.from_messages([("system", system), ("human", human)])

chain = prompt | llm
response = chain.invoke({"text": "How to increase inference speed for a 7B LLM?"})

CPU times: user 44.7 ms, sys: 36.8 ms, total: 81.4 ms
Wall time: 2.58 s


In [17]:
print(response.content)

As a seasoned Machine Learning & AI Engineer, I'd be happy to help you optimize the inference speed for a 7B Large Language Model (LLM).

Here are some strategies to increase inference speed for a 7B LLM:

1. **Model Pruning**: Remove redundant or unnecessary weights and connections in the model to reduce its size and computational requirements. This can be done using techniques like magnitude-based pruning, L1 regularization, or iterative pruning.
2. **Knowledge Distillation**: Train a smaller, simpler model (the student) to mimic the behavior of the large, pre-trained model (the teacher). This can reduce the model size and inference time while preserving the original model's performance.
3. **Quantization**: Represent the model's weights and activations using fewer bits (e.g., int8 instead of float32). This reduces memory usage and can lead to faster inference times. Techniques like post-training quantization, quantization-aware training, or dynamic quantization can be used.
4. **Ten

## Agents

In [8]:
customer_communicator = Agent(
    role="Senior cryptocurrency customer communicator",
    goal="Find which cryptocurrency the customer is interested in",
    backstory="""You're highly experienced in communicating about cryptocurrencies
    and blockchain technology with customers and their research needs""",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=human_tools,
)

In [9]:
news_analyst = Agent(
    role="Cryptocurrency News Analyst",
    goal="""Get news for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency news.
    You have a complete understanding of macroeconomic factors, but you specialize
    into analyzing news.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_news_tool],
)

price_analyst = Agent(
    role="Cryptocurrency Price Analyst",
    goal="""Get historical prices for a given cryptocurrency. Write 1 paragraph analysis of
    the market and make prediction - up, down or neutral.""",
    backstory="""You're an expert analyst of trends based on cryptocurrency
    historical prices. You have a complete understanding of macroeconomic factors,
    but you specialize into technical analys based on historical prices.
    """,
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=True,
    tools=[cryptocurrency_price_tool],
)

In [10]:
writer = Agent(
    role="Cryptocurrency Report Writer",
    goal="""Write 1 paragraph report of the cryptocurrency market.""",
    backstory="""
    You're widely accepted as the best cryptocurrency analyst that
    understands the market and have tracked every asset for more than 10 years. Your trends
    analysis are always extremely accurate.

    You're also master level analyst in the traditional markets and have deep understanding
    of human psychology. You understand macro factors and combine those multiple
    theories - e.g. cycle theory. You're able to hold multiple opininons when analysing anything.

    You understand news and historical prices, but you look at those with a
    healthy dose of skepticism. You also consider the source of news articles.

    Your most well developed talent is providing clear and concise summarization
    that explains very complex market topics in simple to understand terms.

    Some of your writing techniques include:

    - Creating a bullet list (executive summary) of the most importannt points
    - Distill complex analyses to their most important parts

    You writing transforms even dry and most technical texts into
    a pleasant and interesting read.""",
    llm=llm,
    verbose=True,
    max_iter=5,
    memory=True,
    allow_delegation=False,
)

## Tasks

In [11]:
get_cryptocurrency = Task(
    description=f"Ask which cryptocurrency the customer is interested in.",
    expected_output="""Cryptocurrency symbol that the human wants you to research e.g. BTC.""",
    agent=customer_communicator,
)

In [12]:
get_news_analysis = Task(
    description=f"""
    Use the search tool to get news for the cryptocurrency

    The current date is {datetime.now()}.

    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph report for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=news_analyst,
    context=[get_cryptocurrency],
)


get_price_analysis = Task(
    description=f"""
    Use the price tool to get historical prices

    The current date is {datetime.now()}.

    Compose the results into a helpful report""",
    expected_output="""Create 1 paragraph summary for the cryptocurrency,
    along with a prediction for the future trend
    """,
    agent=price_analyst,
    context=[get_cryptocurrency],
)

In [13]:
write_report = Task(
    description=f"""Use the reports from the news analyst and the price analyst to
    create a report that summarizes the cryptocurrency""",
    expected_output="""1 paragraph report that summarizes the market and
    predicts the future prices (trend) for the cryptocurrency""",
    agent=writer,
    context=[get_news_analysis, get_price_analysis],
)

## Crew

In [14]:
crew = Crew(
    agents=[customer_communicator, price_analyst, news_analyst, writer],
    tasks=[get_cryptocurrency, get_news_analysis, get_price_analysis, write_report],
    verbose=2,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)

In [15]:
results = crew.kickoff()

 [DEBUG]: == Working Agent: Senior cryptocurrency customer communicator
 [INFO]: == Starting Task: Ask which cryptocurrency the customer is interested in.


> Entering new CrewAgentExecutor chain...
Thought: I need to ask the customer which cryptocurrency they are interested in.

Action: human
Action Input: {"question": "Hello! I'm here to help you with your cryptocurrency research. Which cryptocurrency are you interested in?"}

Hello! I'm here to help you with your cryptocurrency research. Which cryptocurrency are you interested in?
Bitcoin
 

Bitcoin

Thought: I think I have the answer, but I want to confirm.

Action: human
Action Input: {"question": "Just to confirm, you're interested in Bitcoin, correct?"}

Just to confirm, you're interested in Bitcoin, correct?
Correct
 

Correct

Thought: I now know the final answer
Final Answer: BTC

> Finished chain.
 [DEBUG]: == [Senior cryptocurrency customer communicator] Task output: BTC


 [DEBUG]: == Working Agent: Cryptocurrency News Ana